# K-Means Cluster Feature Engineering (Task 1)

Engineers categorical cluster tags from the 8 scaled intelligence features by fitting a K-Means model and one-hot encoding each sample's assigned cluster.
These cluster tags are consumed by the multinomial logistic-regression baseline as extra features, alongside the Gaussian noise-augmented intelligence scores.

The KMeans model is fit on the training split only and then used to tag both train and test samples, so no test information leaks into the cluster geometry.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# Add the parent directory to the system path so we can import preprocess utilities
sys.path.append(os.path.abspath('..'))
from preprocess import clean_raw_data, preprocess_data, rename_columns

# One cluster per multiple-intelligence dimension
DEFAULT_N_CLUSTERS = 8

## Define K-Means Fitting Function

In [ ]:
def fit_kmeans(X: pd.DataFrame, n_clusters: int = DEFAULT_N_CLUSTERS, random_state: int = 42) -> KMeans:
    """Fit a K-Means model on the (scaled, noise-augmented) feature matrix."""
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X)
    return kmeans

## Define Feature Appending Function

In [ ]:
def add_cluster_features(X: pd.DataFrame, kmeans: KMeans) -> pd.DataFrame:
    """Append one-hot encoded K-Means cluster tags to the feature matrix.

    Returns a new DataFrame with the original features plus ``cluster_0`` …
    ``cluster_{k-1}`` indicator columns. The index of ``X`` is preserved so the
    result stays aligned with the corresponding labels.
    """
    labels = kmeans.predict(X)
    n_clusters = kmeans.n_clusters

    cluster_cols = [f'cluster_{i}' for i in range(n_clusters)]
    one_hot = pd.DataFrame(
        np.eye(n_clusters, dtype=float)[labels],
        columns=cluster_cols,
        index=X.index,
    )
    return pd.concat([X, one_hot], axis=1)

## Demonstrate Cluster Feature Generation
We load the raw dataset, clean and preprocess it, fit a K-Means model, and append the cluster features.

In [ ]:
# Load dataset
dataset_path = '../dataset/dataset_skill_predictor.csv'
df = pd.read_csv(dataset_path)
df = clean_raw_data(df)

if 'Job profession' in df.columns:
    df = rename_columns(df)

X, y, label_encoder = preprocess_data(df)
print(f"Original feature shape: {X.shape}")

# Fit KMeans
kmeans = fit_kmeans(X, n_clusters=DEFAULT_N_CLUSTERS, random_state=42)

# Add cluster features
X_augmented = add_cluster_features(X, kmeans)
print(f"Augmented feature shape: {X_augmented.shape}")
X_augmented.head()